# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [58]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0,0)
        self.walls = {(0,3), (1,1), (2,4), (4,2)}
        self.slippery_states = {(1,2), (3,1), (4,3)}

        self.terminal_states = {
            (0, 5): 10.0,
            (2,2): 2.0
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
            (3, 5): -10.0
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    # Valida que no se salga del los limtes del mapa y que no choque con una pared
    def is_valid_state(self, state):
        if(state[0] < 0 or state[0] >= self.height or
           state[1] < 0 or state[1] >= self.width or
           state in self.walls):
            return False
        return True

    # Devuelve todos los estados posibles del mapa, incertando los validos dentro de un arreglo
    def states(self):
        states = []
        for i in range(self.height):
            for j in range(self.width):
                if self.is_valid_state((i,j)):
                    states.append((i,j))
        return states
 
    # Valida que el estado que estamos revisando sea o una terminal con recompensa o un estado de peligro con reduccion de recompensa
    def is_terminal(self, state):
        return state in self.terminal_states

    # Si el estado es terminal, valida de que tipo es y devuelve la recompensa o penalizacion correspondiente, 
    # de lo contrario aplica la penalizacion por vivir de -1
    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        elif state in self.danger_states:
            return self.danger_states[state]
        else:
            return self.living_reward

    def get_transition_probs(self, state, action):

        #Si nos encontramos en un estado terminal, no hay transiciones posibles
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Definimos el siguiente estado como la suma del estado actual y la accion que se esta tomando y validamos que sea valida
        # de lo contrario no se movera
        next_state = (state[0] + action[0], state[1] + action[1])
        if not self.is_valid_state(next_state):
                next_state = state

        # Definimos las acciones de los posibles giros que puede tomar el agente y sus correspondientes estados,
        #  nuevamente validamos que sea un estado valido, de lo contrario no se movera
        left_action = (-action[1], action[0])  # Acción a la izquierda
        right_action = (action[1], -action[0])  # Acción a la derecha

        left_state = (state[0] + left_action[0], state[1] + left_action[1])
        if not self.is_valid_state(left_state):
            left_state = state

        right_state = (state[0] + right_action[0], state[1] + right_action[1])
        if not self.is_valid_state(right_state):
            right_state = state


        #Si el piso es resbaloso en ese estado, tiene una probabilidad de 0.6 de ir al estado deseado, y 0.4 de girar
        if state in self.slippery_states:
            return [
                (next_state, 0.6),
                (left_state, 0.2),
                (right_state, 0.2)
            ]

        #Si el piso no es resbaloso, tiene una probabilidad de 0.9 de ir al estado deseado, y 0.1 de girar
        else:
            return [
                (next_state, 0.9),
                (left_state, 0.05),
                (right_state, 0.05)
            ]



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [53]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [61]:
def expected_next_value(grid, state, action, V):
    # Suma los valores esperados de los estados siguientes multiplicados por sus probabilidades de transición.
    V_esperado = 0
    for next_state, prob in grid.get_transition_probs(state, action):
        V_esperado += prob * V[next_state]
    return V_esperado


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # Inicializa el valor de todos los estados en 0
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        delta = 0.0 # Variable para rastrear el cambio máximo en los valores de los estados
        new_V = V.copy() # Copia de los valores originales para no modificar mientras iteramos

        for s in grid.states():
            # Si el estado es terminal, no se actualiza su valor
            if grid.is_terminal(s):
                continue

            # Calcula el valor esperado para cada acción y selecciona la mejor
            action_values = []
            for a in grid.actions:
                expected_value = expected_next_value(grid, s, a, V)
                action_values.append(expected_value)

            best_action_value = max(action_values)

            # Actualiza el valor del estado con la recompensa inmediata más el valor esperado descontado de la mejor acción
            new_V[s] = grid.get_reward(s) + grid.gamma * best_action_value 

            delta = max(delta, abs(new_V[s] - V[s])) # Actualiza el cambio máximo en los valores de los estados

        V = new_V # Actualiza los valores de los estados para la siguiente iteración

        # Si el cambio máximo en los valores de los estados es menor que el umbral (threshold), la iteración ha convergido
        if delta < threshold:
            break

    return V, i # Devuelve los valores de los estados y el número de iteraciones realizadas


def extract_policy(grid, V):
    policy = {} 

    # Recorre todos los estados del mapa y para cada estado no terminal, calculando la acción que maximiza el valor esperado de los estados siguientes.
    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        best_value = float('-inf')
        for a in grid.actions:
            expected_value = expected_next_value(grid, s, a, V)
            if expected_value > best_value:
                best_value = expected_value
                best_action = a

        policy[s] = best_action # Asigna la mejor acción encontrada para el estado actual en la política

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [62]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # Inicializamos todos los valores de los estados en 0
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        delta = 0.0 # Variable para rastrear el cambio máximo en los valores de los estados
        new_V = V.copy() # Copia de los valores originales para no modificar mientras iteramos

        for s in grid.states():
            # Si el estado es terminal, no se actualiza su valor
            if grid.is_terminal(s):
                continue

            # Obtiene la acción de la política para el estado actual
            a = policy[s]

            # Calcula el valor esperado para la acción de la política
            expected_value = expected_next_value(grid, s, a, V)

            # Actualiza el valor del estado con la recompensa inmediata más el valor esperado descontado de la acción de la política
            new_V[s] = grid.get_reward(s) + grid.gamma * expected_value 

            delta = max(delta, abs(new_V[s] - V[s])) # Actualiza el cambio máximo en los valores de los estados

        V = new_V # Actualiza los valores de los estados para la siguiente iteración

        # Si el cambio máximo en los valores de los estados es menor que el umbral (threshold), la evaluación ha convergido
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    policy = {}

    # Recorre todos los estados del mapa y para cada estado no terminal, calculando la acción que maximiza el valor esperado de los estados siguientes.
    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        best_value = float('-inf')
        for a in grid.actions:
            expected_value = expected_next_value(grid, s, a, V)
            if expected_value > best_value:
                best_value = expected_value
                best_action = a

        policy[s] = best_action # Asigna la mejor acción encontrada para el estado actual en la política

    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # Inicializa una política aleatoria para todos los estados no terminales
    policy = {s: grid.actions[np.random.choice(len(grid.actions))] for s in grid.states() if not grid.is_terminal(s)}

    for i in range(max_iter):
        # Evaluación de la política actual
        V = policy_evaluation(grid, policy, threshold)

        # Mejora de la política basada en los valores obtenidos
        new_policy = policy_improvement(grid, V)

        # Si la política no cambia, hemos convergido y podemos salir del bucle
        if new_policy == policy:
            break

        policy = new_policy

    return policy, V, i



## Parte 4 — Visualización y comparación


In [65]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [66]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 15

Valores:
 -3.839 |  -3.857 |  -3.099 |   WALL   |  -1.243 |  +0.000
 -3.077 |   WALL   |  -2.205 |  -2.258 |  -4.165 |  -1.243
 -2.222 |  -1.168 |  +0.000 |  -1.199 |   WALL   |  -2.206
 -3.053 |  -2.572 |  -1.213 |  -2.164 |  -3.065 | -12.486
 -3.896 |  -5.507 |   WALL   |  -3.871 |  -3.879 |  -4.925

Política:
 ↓  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  ↓  |  ↑  |  ↑ 
 →  |  →  | +2 |  ←  |  #  |  ↑ 
 ↑  |  ↑  |  ↑  |  ↑  |  ←  |  ↑ 
 ↑  |  ↑  |  #  |  ↑  |  ↑  |  ← 

=== POLICY ITERATION ===
Historia: 3

Valores:
 -3.839 |  -3.857 |  -3.099 |   WALL   |  -1.243 |  +0.000
 -3.077 |   WALL   |  -2.205 |  -2.258 |  -4.165 |  -1.243
 -2.222 |  -1.168 |  +0.000 |  -1.199 |   WALL   |  -2.206
 -3.053 |  -2.572 |  -1.213 |  -2.164 |  -3.065 | -12.486
 -3.896 |  -5.507 |   WALL   |  -3.871 |  -3.879 |  -4.925

Política:
 ↓  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  ↓  |  ↑  |  ↑ 
 →  |  →  | +2 |  ←  |  #  |  ↑ 
 ↑  |  ↑  |  ↑  |  ↑  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?


### Respuestas
1. El robot buscara la recompensa de +2 antes de la entrega de +10 ya que esta mucho mas cercano al punto de inicio ademas de mas seguro que ir al +10.
2.  Una recompensa menor puede ser mas optima cuando la recompensa mayor esta mas lejano y el agente acumula muchas penalizaciones.
3.  La desicion cambia en los estados vecinos en donde se decide si vale la pena arriesgarse a pasar por las casillas resbaladizas.
4.  El costo por paso fuerza al agente a encontrar una solucion rapido y evitar que se quede realizando pasos redundantes dentro del mapa.
5.  Porque el modelo de transicion varia dependiendo de si el piso es resbaladiso o no, forzando a T a consultar su estado antes de decidir que probabilidad usar

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

R= No debe ocurrir mayor cambio dentro de la politica dado que de una forma u otra se esta penando al agente por moverse

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

R= Ocurrio un aumento del numero de iteraciones

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

R= No valoro mas la recompensa ya que al aumentar el gamma peso mas el costo acumulado de los pasos ya que la distancia era mucha, por lo que dio prioridad a la recompensa del +2

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
